<a href="https://colab.research.google.com/github/m-uu-dacss-690c/Homework_3/blob/main/index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

AHP in Python: Multicriteria Decision-Making

The Saaty Son School Decision

In [1]:
# necessary libraries
import pandas as pd
import networkx as nx
import numpy as np
!pip install AHPy # not installed by default in colab
import ahpy


In [2]:
# read in data
ahp_data_link = "https://github.com/m-uu-dacss-690c/Homework_3/raw/refs/heads/main/ahp_comparisons.xlsx"
# ahp_data_link = "https://github.com/m-uu-dacss-690c/Homework_3/raw/refs/heads/main/saaty_style_ahp_comparisons.xlsx"

# convert each sheet in excel file to dataframe as adjacency matrices
pairwise_criteria = pd.read_excel(ahp_data_link, sheet_name="criteria", index_col=0)
pairwise_learning = pd.read_excel(ahp_data_link, sheet_name="learning", index_col=0)
pairwise_friends = pd.read_excel(ahp_data_link, sheet_name="friends", index_col=0)
pairwise_school_life = pd.read_excel(ahp_data_link, sheet_name="school_life", index_col=0)
pairwise_vocational_training = pd.read_excel(ahp_data_link, sheet_name="vocational_training", index_col=0)
pairwise_college_prep = pd.read_excel(ahp_data_link, sheet_name="college_prep", index_col=0)
pairwise_music_classes = pd.read_excel(ahp_data_link, sheet_name="music_classes", index_col=0)



Pairwise Criteria Matrix

In [3]:
pairwise_criteria

,learning,friends,school_life,vocational_training,college_prep,music_classes
learning,NaN,4.0,3.0,1.0,3.0,4.0
friends,NaN,NaN,7.0,3.0,NaN,1.0
school_life,NaN,NaN,NaN,NaN,NaN,NaN
vocational_training,1.0,NaN,5.0,NaN,1.0,NaN
college_prep,NaN,5.0,5.0,1.0,NaN,3.0
music_classes,NaN,1.0,6.0,3.0,NaN,NaN


In [4]:
# turn adjacency matrices into pairwise comparisons, compatible with ahp lib
# also, ignore NAs

# convert pandas data frame to directed graph object
g_criteria = nx.from_pandas_adjacency(pairwise_criteria, create_using=nx.MultiDiGraph)
# dictionary comprehension to get source, target node, and weight and put them in a dict
criteria_comparisons = {(e[0],e[1]):e[2]['weight'] for e in g_criteria.edges(data=True) if np.isfinite(e[2]['weight'])}

g_learning = nx.from_pandas_adjacency(pairwise_learning, create_using=nx.MultiDiGraph)
learning_comparisons = {(e[0],e[1]):e[2]['weight'] for e in g_learning.edges(data=True) if np.isfinite(e[2]['weight'])}

g_friends = nx.from_pandas_adjacency(pairwise_friends, create_using=nx.MultiDiGraph)
friends_comparisons = {(e[0],e[1]):e[2]['weight'] for e in g_friends.edges(data=True) if np.isfinite(e[2]['weight'])}

g_school_life = nx.from_pandas_adjacency(pairwise_school_life, create_using=nx.MultiDiGraph)
school_life_comparisons = {(e[0],e[1]):e[2]['weight'] for e in g_school_life.edges(data=True) if np.isfinite(e[2]['weight'])}

g_vocational_training = nx.from_pandas_adjacency(pairwise_vocational_training, create_using=nx.MultiDiGraph)
vocational_training_comparisons = {(e[0],e[1]):e[2]['weight'] for e in g_vocational_training.edges(data=True) if np.isfinite(e[2]['weight'])}

g_college_prep = nx.from_pandas_adjacency(pairwise_college_prep, create_using=nx.MultiDiGraph)
college_prep_comparisons = {(e[0],e[1]):e[2]['weight'] for e in g_college_prep.edges(data=True) if np.isfinite(e[2]['weight'])}

g_music_classes = nx.from_pandas_adjacency(pairwise_music_classes, create_using=nx.MultiDiGraph)
music_classes_comparisons = {(e[0],e[1]):e[2]['weight'] for e in g_music_classes.edges(data=True) if np.isfinite(e[2]['weight'])}

In [5]:
# create ahpy objects from comparison dicts
criteria = ahpy.Compare('criteria', criteria_comparisons, random_index='saaty')
learning = ahpy.Compare('learning', learning_comparisons, random_index='saaty')
friends = ahpy.Compare('friends', friends_comparisons, random_index='saaty')
school_life = ahpy.Compare('school_life', school_life_comparisons, random_index='saaty')
vocational_training = ahpy.Compare('vocational_training', vocational_training_comparisons, random_index='saaty')
college_prep = ahpy.Compare('college_prep', college_prep_comparisons, random_index='saaty')
music_classes = ahpy.Compare('music_classes', music_classes_comparisons, random_index='saaty')



In [6]:
# create hierarchy
criteria.add_children([learning, friends, school_life, vocational_training, college_prep, music_classes])

In [7]:
# view importance of each criterion
weights=criteria.global_weights
pd.Series(weights).to_frame(name='Value')

,Value
learning,0.3208
college_prep,0.2374
friends,0.1395
music_classes,0.1391
vocational_training,0.1285
school_life,0.0348


In [8]:
# view how much each school satisfies weighted criteria
priorities=criteria.target_weights
pd.Series(priorities).to_frame(name='Value')

,Value
haverford (b),0.3785
lower merian (a),0.3674
harrington (c),0.2542


Haverford is the best option by a tiny margin

In [9]:
# check to ensure consistency is sufficient
# if below 0.1, decisions are considered transitive enough for a valid conclusion
assesment=[(val.name,val.consistency_ratio) for val in [criteria, learning, college_prep, friends, music_classes, vocational_training, school_life]]
pd.DataFrame(assesment,columns=['Criterion', 'Consistency Ratio'])


,Criterion,Consistency Ratio
0,criteria,0.2272
1,learning,0.0516
2,college_prep,0.0000
3,friends,0.0000
4,music_classes,0.0516
5,vocational_training,0.2005
6,school_life,0.0000


# Conclusion
Since the criteria and vocational training have a consistency ratio of about 0.2, higher than the allowable 0.1, the solution should not be relied upon. The decisions made when creating the original comparison preference weights were insufficently transitive.

This is true if the ahp comparison excel file is formatted in the style of saaty's video (fully filled using 1s for self to self comparisons, and fractions for the losing side of each comparison), or in the style used in the slides (leaving out self to self comparisons and the losing side of each comparison).